# Notebook 03 - LSTM Training

**Kelompok 08 - Teknik Komputer UI**

| NPM | Nama |
|-----|------|
| 2306242395 | Calvin Wirathama Katoroy |
| 2306220532 | Syahmi Hamdani |
| 2306220323 | Christover Angelo |
| 2306221024 | Matthew Immanuel |

**Tujuan:** Melatih model LSTM sebagai main model dengan 6 eksperimen hyperparameter (hidden_size, num_layers, dropout, seq_len). History training disimpan ke Drive bersama checkpoint agar loss curve tersedia di sesi Colab berikutnya.

In [ ]:
import subprocess, os
from pathlib import Path

REPO_URL  = 'https://github.com/calvinkatoroy/tugas-akhir-ai-kel-08.git'
REPO_NAME = 'tugas-akhir-ai-kel-08'

cwd = Path.cwd()

if (cwd / '../src').resolve().exists():
    result = subprocess.run(['git', '-C', str((cwd / '..').resolve()), 'pull'],
                            capture_output=True, text=True)
    print(result.stdout.strip() or 'Already up to date.')

elif (cwd / 'src').exists():
    result = subprocess.run(['git', 'pull'], capture_output=True, text=True)
    print(result.stdout.strip() or 'Already up to date.')
    os.chdir('notebooks')

else:
    repo_path = cwd / REPO_NAME
    if not repo_path.exists():
        print(f'Cloning {REPO_URL} ...')
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    else:
        print('Repo found, pulling latest...')
        subprocess.run(['git', '-C', str(repo_path), 'pull'], capture_output=True)
    os.chdir(repo_path / 'notebooks')

print(f'Working dir: {Path.cwd()}')

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings, random, copy
import numpy as np
import torch
import yaml
from pathlib import Path

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
    print('Mixed precision: ON (autocast + GradScaler)')

def safe_log_experiment(record, csv_path):
    import os, pandas as pd
    from pathlib import Path
    path = Path(csv_path)
    if path.exists():
        try:
            pd.read_csv(path)
        except Exception:
            print(f'WARNING: corrupt CSV at {path}, resetting.')
            os.remove(path)
    from src.utils import log_experiment
    log_experiment(record, csv_path=csv_path)


In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/tugas-akhir-ai")
    SPLITS_DIR = DRIVE_ROOT / "splits"
else:
    import yaml
    with open("../config.yaml") as f:
        _cfg = yaml.safe_load(f)
    DRIVE_ROOT = Path(_cfg["data"]["drive_root"])
    SPLITS_DIR = Path(_cfg["data"]["splits_drive"])

print(f"DRIVE_ROOT: {DRIVE_ROOT}")
print(f"SPLITS_DIR: {SPLITS_DIR}")

In [ ]:
with open("../config.yaml") as f:
    cfg = yaml.safe_load(f)

X_train = np.load(SPLITS_DIR / "X_train_seq.npy")
y_train = np.load(SPLITS_DIR / "y_train_seq.npy")
X_val   = np.load(SPLITS_DIR / "X_val_seq.npy")
y_val   = np.load(SPLITS_DIR / "y_val_seq.npy")
X_test  = np.load(SPLITS_DIR / "X_test_seq.npy")
y_test  = np.load(SPLITS_DIR / "y_test_seq.npy")

n_features = X_train.shape[2]
seq_len    = X_train.shape[1]
print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")
print(f"n_features={n_features}, seq_len={seq_len}")

## Baseline run (config defaults)

In [ ]:
from src.models.lstm_model import build_lstm
from src.train import train_model
from src.utils import log_experiment

model = build_lstm(cfg, n_features)
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total_params:,}')

In [ ]:
import os, shutil, json

_ckpt_local = '../results/checkpoints/best_lstm.pt'
_ckpt_drive = str(DRIVE_ROOT / 'best_lstm.pt')
_hist_local = '../results/checkpoints/history_lstm_01.json'
_hist_drive = str(DRIVE_ROOT / 'history_lstm_01.json')

os.makedirs('../results/checkpoints', exist_ok=True)
if not os.path.exists(_ckpt_local) and os.path.exists(_ckpt_drive):
    shutil.copy(_ckpt_drive, _ckpt_local)
    print('Restored best_lstm.pt from Drive')
    if os.path.exists(_hist_drive):
        shutil.copy(_hist_drive, _hist_local)
        print('Restored history from Drive')

if os.path.exists(_ckpt_local):
    print('Checkpoint found - skipping baseline training.')
    if os.path.exists(_hist_local):
        with open(_hist_local) as f:
            history_baseline = json.load(f)
        print('History loaded.')
    elif os.path.exists(_hist_drive):
        with open(_hist_drive) as f:
            history_baseline = json.load(f)
        print('History loaded from Drive.')
    else:
        history_baseline = {'val_f1': [0.0], 'train_loss': [], 'val_loss': []}
        print('No saved history - loss curve unavailable this session.')
else:
    history_baseline, ckpt_baseline = train_model(
        model, X_train, y_train, X_val, y_val,
        cfg=cfg, model_key='lstm',
        checkpoint_dir='../results/checkpoints',
    )
    with open(_hist_local, 'w') as f:
        json.dump({k: [float(v) for v in vals]
                   for k, vals in history_baseline.items()}, f)
    shutil.copy(_hist_local, _hist_drive)
    print('History saved to Drive.')
    safe_log_experiment({
        'exp_id': 'lstm_01_baseline',
        'model': 'LSTM',
        'hidden_size': cfg['lstm']['hidden_size'],
        'num_layers': cfg['lstm']['num_layers'],
        'dropout': cfg['lstm']['dropout'],
        'seq_len': cfg['lstm']['seq_len'],
        'lr': cfg['lstm']['learning_rate'],
        'batch_size': cfg['lstm']['batch_size'],
        'best_val_f1': round(max(history_baseline['val_f1']), 4),
        'notes': 'baseline - config defaults',
    }, csv_path=str(DRIVE_ROOT / 'metrics_summary.csv'))


## Hyperparameter Experiments
Document at least 5 runs. Change ONE hyperparam at a time.

In [ ]:
def run_lstm_experiment(exp_id, overrides, notes=''):
    """overrides: dict of cfg['lstm'] keys to override."""
    exp_cfg = copy.deepcopy(cfg)
    exp_cfg['lstm'].update(overrides)

    random.seed(SEED); np.random.seed(SEED)
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

    if torch.cuda.is_available(): torch.cuda.empty_cache()
    m = build_lstm(exp_cfg, n_features)
    hist, ckpt = train_model(
        m, X_train, y_train, X_val, y_val,
        cfg=exp_cfg, model_key='lstm',
        checkpoint_dir='../results/checkpoints',
    )
    best_f1 = round(max(hist['val_f1']), 4)
    safe_log_experiment({
        'exp_id': exp_id,
        'model': 'LSTM',
        'hidden_size': exp_cfg['lstm']['hidden_size'],
        'num_layers': exp_cfg['lstm']['num_layers'],
        'dropout': exp_cfg['lstm']['dropout'],
        'seq_len': exp_cfg['lstm']['seq_len'],
        'lr': exp_cfg['lstm']['learning_rate'],
        'batch_size': exp_cfg['lstm']['batch_size'],
        'best_val_f1': best_f1,
        'notes': notes,
    }, csv_path=str(DRIVE_ROOT / "metrics_summary.csv"))
    print(f'[{exp_id}] best_val_f1={best_f1}')
    return hist, ckpt, best_f1

In [ ]:
# Exp 02 - smaller hidden size
hist_02, ckpt_02, f1_02 = run_lstm_experiment(
    'lstm_02_hidden64', {'hidden_size': 64}, 'hidden_size=64')

In [ ]:
# Exp 03 - larger hidden size
hist_03, ckpt_03, f1_03 = run_lstm_experiment(
    'lstm_03_hidden256', {'hidden_size': 256}, 'hidden_size=256')

In [ ]:
# Exp 04 - 3 layers
hist_04, ckpt_04, f1_04 = run_lstm_experiment(
    'lstm_04_layers3', {'num_layers': 3}, 'num_layers=3')

In [ ]:
# Exp 05 - higher dropout
hist_05, ckpt_05, f1_05 = run_lstm_experiment(
    "lstm_05_dropout05", {"dropout": 0.5}, "dropout=0.5")

In [ ]:
# Exp 06 - shorter sequence window
# NOTE: requires rebuilding sequences at seq_len=5
# Load the flat splits and re-window
from src.preprocessing import make_sequences
import joblib

X_tr_flat = np.load(SPLITS_DIR / 'X_train.npy')
y_tr_flat = np.load(SPLITS_DIR / 'y_train.npy')
X_vl_flat = np.load(SPLITS_DIR / 'X_val.npy')
y_vl_flat = np.load(SPLITS_DIR / 'y_val.npy')

X_tr5, y_tr5 = make_sequences(X_tr_flat, y_tr_flat, seq_len=5)
X_vl5, y_vl5 = make_sequences(X_vl_flat, y_vl_flat, seq_len=5)

exp_cfg6 = copy.deepcopy(cfg)
exp_cfg6['lstm']['seq_len'] = 5
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
m6 = build_lstm(exp_cfg6, n_features)
hist_06, ckpt_06 = train_model(
    m6, X_tr5, y_tr5, X_vl5, y_vl5,
    cfg=exp_cfg6, model_key='lstm',
    checkpoint_dir='../results/checkpoints',
)
f1_06 = round(max(hist_06['val_f1']), 4)
safe_log_experiment({
    'exp_id': 'lstm_06_seqlen5',
    'model': 'LSTM',
    'hidden_size': 128, 'num_layers': 2, 'dropout': 0.3,
    'seq_len': 5, 'lr': 0.001, 'batch_size': cfg['lstm']['batch_size'],
    'best_val_f1': f1_06, 'notes': 'seq_len=5',
}, csv_path=str(DRIVE_ROOT / "metrics_summary.csv"))
print(f'[lstm_06_seqlen5] best_val_f1={f1_06}')

## Final Evaluation on Test Set

In [ ]:
import pandas as pd

results = pd.read_csv(str(DRIVE_ROOT / 'metrics_summary.csv'))
lstm_results = results[results['model'] == 'LSTM'].copy()
avail_cols = [c for c in ['exp_id', 'hidden_size', 'num_layers', 'dropout', 'seq_len', 'lr', 'best_val_f1', 'notes'] if c in lstm_results.columns]
print(lstm_results[avail_cols])

best_row = lstm_results.loc[lstm_results['best_val_f1'].idxmax()]
print(f'\nBest: {best_row["exp_id"]}  val_F1={best_row["best_val_f1"]}')

_hist_map = {
    'lstm_01_baseline': history_baseline,
    'lstm_02_hidden64': hist_02,
    'lstm_03_hidden256': hist_03,
    'lstm_04_layers3': hist_04,
    'lstm_05_dropout05': hist_05,
    'lstm_06_seqlen5': hist_06,
}
best_history = _hist_map.get(best_row['exp_id'], history_baseline)
if not best_history.get('train_loss'):
    for _name in ['hist_02', 'hist_03', 'hist_04', 'hist_05', 'hist_06']:
        _h = globals().get(_name)
        if _h and _h.get('train_loss'):
            best_history = _h
            print(f'Using fallback history from {_name} (baseline not available).')
            break


In [ ]:
import copy
from src.models.lstm_model import build_lstm
from src.evaluate import evaluate_dl_model
from src.utils import plot_loss_curves

best_cfg = copy.deepcopy(cfg)
best_cfg['lstm']['hidden_size'] = int(best_row['hidden_size'])
best_cfg['lstm']['num_layers']  = int(best_row['num_layers'])
best_cfg['lstm']['dropout']     = float(best_row['dropout'])

best_model = build_lstm(best_cfg, n_features).to(device)
best_model.load_state_dict(torch.load('../results/checkpoints/best_lstm.pt', map_location=device))
best_model.eval()

y_pred_lstm, y_prob_lstm = evaluate_dl_model(
    best_model, X_test, y_test,
    model_name='LSTM',
    history=best_history,
    figures_dir='../results/figures',
    csv_path=str(DRIVE_ROOT / 'metrics_summary.csv'),
)

np.save('../results/lstm_y_prob.npy', y_prob_lstm)
np.save('../results/y_test_seq.npy', y_test)


In [ ]:
_plot_hist = best_history if best_history.get('train_loss') else None
if not _plot_hist:
    for _name in ['history_baseline', 'hist_02', 'hist_03', 'hist_04', 'hist_05', 'hist_06']:
        _h = globals().get(_name)
        if _h and _h.get('train_loss'):
            _plot_hist = _h
            print(f'Using fallback history from {_name}.')
            break

if _plot_hist:
    plot_loss_curves(
        _plot_hist['train_loss'], _plot_hist['val_loss'],
        title='LSTM Loss Curves (best)',
        save_path='../results/figures/loss_lstm.png',
    )
else:
    print('No training history available - skipping loss curve.')


In [ ]:
# Save best checkpoint to Drive so it persists across sessions
import shutil
shutil.copy("../results/checkpoints/best_lstm.pt", str(DRIVE_ROOT / "best_lstm.pt"))
print(f"Checkpoint saved to Drive: {DRIVE_ROOT / "best_lstm.pt"}")